In [1]:
from tensorflow import keras as tf_keras

train_dataset = tf_keras.utils.text_dataset_from_directory(
    'data-files/aclimdb/train', batch_size=32
)
validation_dataset = tf_keras.utils.text_dataset_from_directory(
    'data-files/aclimdb/val', batch_size=32
)
test_dataset = tf_keras.utils.text_dataset_from_directory(
    'data-files/aclimdb/test', batch_size=32
)

Found 20000 files belonging to 2 classes.
Found 5000 files belonging to 2 classes.
Found 25000 files belonging to 2 classes.


In [9]:
# BoW 모델 기반 텍스트 데이터 인코딩 도구 학습

text_vectorization = tf_keras.layers.TextVectorization(

    ngrams=2,

    max_tokens=20000, # 단어 사전에 포함될 단어 갯수 (빈도수 높은 순)

    # output_mode='int' # 각 단어의 단어 사전에 지정된 번호 인코딩
    # output_mode="multi_hot" # 단어가 있는 곳에 갯수와 관계 없이 1로 인코딩
    # output_mode='count' # 단어가 있는 곳에 갯수를 인코딩
    output_mode='tf-idf' # 단어가 있는 곳에 tf-idf 빈도 값 인코딩
)

only_text_dataset = train_dataset.map(lambda x, y: x)
text_vectorization.adapt( only_text_dataset ) # 학습을 통해 단어 사전 구성

In [ ]:
text_vectorization.get_vocabulary()

In [10]:
# 데이터 셋의 각 데이터에 대해 인코딩 처리

encoded_bigram_train_dataset = train_dataset.map( lambda x, y: (text_vectorization(x), y) )
encoded_bigram_validation_dataset = validation_dataset.map( lambda x, y: (text_vectorization(x), y) )
encoded_bigram_test_dataset = test_dataset.map( lambda x, y: (text_vectorization(x), y) )

In [11]:
# 모델 구조 설계
inputs = tf_keras.layers.Input(shape=(20000, ))
x = tf_keras.layers.Dense(32, activation='relu')(inputs)
x = tf_keras.layers.Dropout(0.5)(x)
outputs = tf_keras.layers.Dense(1, activation='sigmoid')(x)

model = tf_keras.Model(inputs, outputs)

# 모델 학습 설계
model.compile(loss="binary_crossentropy",
              optimizer=tf_keras.optimizers.Adam(),
              metrics=['accuracy'])

# 모델 저장 콜백 만들기
callbacks = [
    tf_keras.callbacks.ModelCheckpoint('models/imdb-dense-model.keras', save_best_only=True)
]

# 모델 훈련
history = model.fit(encoded_bigram_train_dataset, epochs=10, 
                    validation_data=encoded_bigram_validation_dataset, 
                    callbacks=callbacks)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 15s 22ms/step - accuracy: 0.8213 - loss: 0.5073 - val_accuracy: 0.9060 - val_loss: 0.2608
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 14s 22ms/step - accuracy: 0.9146 - loss: 0.2201 - val_accuracy: 0.9036 - val_loss: 0.2431
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 14s 23ms/step - accuracy: 0.9361 - loss: 0.1710 - val_accuracy: 0.8974 - val_loss: 0.2726
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 14s 22ms/step - accuracy: 0.9503 - loss: 0.1315 - val_accuracy: 0.9008 - val_loss: 0.2719
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 14s 22ms/step - accuracy: 0.9602 - loss: 0.1047 - val_accuracy: 0.8996 - val_loss: 0.3084
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 14s 23ms/step - accuracy: 0.9649 - loss: 0.0898 - val_accuracy: 0.8982 - val_loss: 0.3300
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 13s 21ms/step - accuracy: 0.9679 - loss: 0.0834 - val_accuracy: 0.8988 - val_loss: 0.3649
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 12s 20ms/step - accuracy: 0.9693 - loss: 0.0768 - 

In [12]:
best_model = tf_keras.models.load_model('models/imdb-dense-model.keras')
best_model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 20000)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │       640,032 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,920,197 (7.32 MB)

 Trainable params: 640,065 (2.44 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,280,132 (4.88 MB)

In [13]:
best_model.evaluate(encoded_bigram_test_dataset)

782/782 ━━━━━━━━━━━━━━━━━━━━ 10s 13ms/step - accuracy: 0.8918 - loss: 0.2662


[0.26624795794487, 0.8917999863624573]